# 01 — K-Means : classification non supervisée des jardins
**Question :** quels groupes naturels obtient-on à partir des neuf caractéristiques visuelles validées ?

K-Means ne connaît pas les catégories humaines. Les valeurs inconnues (`null`) sont imputées **sur TRAIN uniquement**. Avec 31 jardins, les clusters sont exploratoires et ne constituent pas une preuve de généralisation.


In [ ]:
from pathlib import Path
import os, json, psycopg, pandas as pd
from dotenv import load_dotenv
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "ai-service/plant-ingestion").exists())
load_dotenv(ROOT / ".env", override=False)
load_dotenv(ROOT / ".env.local", override=True)
dsn = os.environ["GARDNAIZE_ML_DATABASE_URL"]
dsn = dsn.replace("postgresql+psycopg://","postgresql://",1)
FEATURES = ("lawn","terrace","pathway","trees","shrubs","fence","low_wall","stairs","slope")
def conn(): return psycopg.connect(dsn)
import numpy as np
from sklearn.impute import SimpleImputer
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from sklearn.decomposition import PCA
import matplotlib.pyplot as plt
with conn() as c:
    with c.cursor() as cur:
        cur.execute("""SELECT g.image_id,g.dataset_split,f.feature_values
        FROM horticulture.garden_image_candidate g JOIN horticulture.ml_garden_features f USING(image_id)
        WHERE g.review_status='ACCEPTED' AND f.reviewed=true ORDER BY g.image_id""")
        rows=cur.fetchall()
assert rows, "Aucun jardin validé"
X=np.array([[np.nan if f.get(k) is None else float(f[k]) for k in FEATURES] for _,_,f in rows],dtype=float)
ids=[r[0] for r in rows]
splits=[r[1] for r in rows]
train=np.array([s=="TRAIN" for s in splits])
assert train.sum()>=3, "Pas assez d'exemples TRAIN"
imp=SimpleImputer(strategy="most_frequent",keep_empty_features=True)
X_train=imp.fit_transform(X[train]); X_all=imp.transform(X)
print("Jardins :",len(rows),"TRAIN :",train.sum(),"Caractéristiques :",len(FEATURES))



In [ ]:
scores={}
for k in range(2,min(7,int(train.sum()))):
    labels=KMeans(n_clusters=k,random_state=42,n_init=10).fit_predict(X_train)
    if 1<len(set(labels))<len(labels):
        scores[k]=silhouette_score(X_train,labels)
print("Silhouette TRAIN par k :",scores)
k=3
km=KMeans(n_clusters=k,random_state=42,n_init=10).fit(X_train)
pred=km.predict(X_all)
display(pd.DataFrame({"image_id":ids,"split":splits,"cluster":pred}).groupby("cluster").agg(n=("image_id","size")))



In [ ]:
pca=PCA(n_components=2,random_state=42)
xy=pca.fit_transform(X_all)
plt.figure(figsize=(8,5))
plt.scatter(xy[:,0],xy[:,1],c=pred,cmap="tab10")
for (x,y),i in zip(xy,ids): plt.annotate(str(i),(x,y),fontsize=8)
plt.title("K-Means jardins — projection PCA (illustration 2D)")
plt.xlabel("Composante principale 1");plt.ylabel("Composante principale 2");plt.show()
print("Attention : PCA sert à visualiser, pas à prouver la qualité des clusters.")

